# Análise exploratória

Agora vou explorar a base consolidada do ETL para responder ao desafio: comparar capitais por função olhando o que foi empenhado versus o que foi pago, evitar comparações injustas usando apenas anos completos e, quando fizer sentido, detalhar o comportamento por subfunção.

A leitura principal aqui é a taxa de execução financeira e a diferença entre empenhado e pago.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

def formato_real(valor):
    if pd.isna(valor):
        return None

    return (
        f"{valor:,.2f}"
        .replace(",", "_")
        .replace(".", ",")
        .replace("_", ".")
    )

pd.set_option("display.float_format", formato_real)

## Carregamento da base

A base usada aqui é o Parquet consolidado produzido pelo ETL. Isso evita reler os CSVs compactados a cada análise e deixa o fluxo mais rápido e reproduzível.

In [2]:
base = pd.read_parquet("../dados_processados/finbra_consolidado.parquet")
base["Ano"] = base["Ano"].astype(int)

base.head()

,Instituição,Cod.IBGE,UF,População,Coluna,Conta,Identificador da Conta,Valor,Ano,Tipo Conta
0,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,Despesas Exceto Intraorçamentárias,siconfi-cor_TotalDespesas,"874.885.274,98",2020,Totais Intraorçamentárias
1,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01 - Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",2020,Função
2,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01.031 - Ação Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",2020,Subfunção
3,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03 - Essencial à Justiça,siconfi-cor_TotalDespesas,"18.822.895,07",2020,Função
4,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03.092 - Representação Judicial e Extrajudicial,siconfi-cor_TotalDespesas,"18.822.895,07",2020,Subfunção


## Cobertura temporal

Antes de comparar capitais entre anos, preciso confirmar quais exercícios estão completos. O próprio desafio avisa que 2025 está parcial.

In [3]:
capitais_por_ano = (
    base.groupby("Ano", as_index=False)
    .agg(capitais=("Cod.IBGE", "nunique"))
    .assign(
        status=lambda dados: dados["capitais"].apply(
            lambda total: "Completo" if total == 26 else "Parcial"
        )
    )
)

anos_completos = capitais_por_ano.loc[capitais_por_ano["capitais"] == 26, "Ano"].tolist()
dados_completos = base.loc[base["Ano"].isin(anos_completos)].copy()

capitais_por_ano

,Ano,capitais,status
0,2020,26,Completo
1,2021,26,Completo
2,2022,26,Completo
3,2023,26,Completo
4,2024,26,Completo
5,2025,11,Parcial


## Classificação das contas

A classificação entre função e subfunção já foi feita no ETL em `transform.py`, então aqui eu apenas separo os recortes analíticos com base em `Tipo Conta`. Isso evita duplicar lógica e mantém o notebook alinhado ao pipeline de transformação.

In [4]:
dados_completos["Tipo Conta"] = dados_completos["Tipo Conta"].fillna("Outros")

base_funcoes = dados_completos.loc[dados_completos["Tipo Conta"] == "Função"].copy()
base_subfuncoes = dados_completos.loc[dados_completos["Tipo Conta"] == "Subfunção"].copy()

base_funcoes.head()

,Instituição,Cod.IBGE,UF,População,Coluna,Conta,Identificador da Conta,Valor,Ano,Tipo Conta
1,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01 - Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",2020,Função
3,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03 - Essencial à Justiça,siconfi-cor_TotalDespesas,"18.822.895,07",2020,Função
5,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,04 - Administração,siconfi-cor_TotalDespesas,"82.258.800,45",2020,Função
13,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,06 - Segurança Pública,siconfi-cor_TotalDespesas,"3.304.806,91",2020,Função
16,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,08 - Assistência Social,siconfi-cor_TotalDespesas,"22.908.000,00",2020,Função


## Execução financeira

A comparação principal é entre empenhado e pago. Também calculo a taxa de execução financeira e o valor per capita para deixar a leitura mais justa entre capitais de tamanhos diferentes.

In [5]:
execucao_funcoes = (
    base_funcoes.pivot_table(
        index=["Ano", "UF", "Cod.IBGE", "Instituição", "População", "Conta"],
        columns="Coluna",
        values="Valor",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for coluna in [
    "Despesas Empenhadas",
    "Despesas Liquidadas",
    "Despesas Pagas",
    "Inscrição de Restos a Pagar Não Processados",
    "Inscrição de Restos a Pagar Processados",
]:
    if coluna not in execucao_funcoes.columns:
        execucao_funcoes[coluna] = 0

execucao_funcoes["taxa_execucao_pct"] = (
    execucao_funcoes["Despesas Pagas"]
    .div(execucao_funcoes["Despesas Empenhadas"])
    .where(execucao_funcoes["Despesas Empenhadas"] > 0)
    * 100
)
execucao_funcoes["empenhado_per_capita"] = execucao_funcoes["Despesas Empenhadas"].div(execucao_funcoes["População"])
execucao_funcoes["pago_per_capita"] = execucao_funcoes["Despesas Pagas"].div(execucao_funcoes["População"])
execucao_funcoes["saldo_nao_pago"] = execucao_funcoes["Despesas Empenhadas"] - execucao_funcoes["Despesas Pagas"]

execucao_funcoes.head()

Coluna,Ano,UF,Cod.IBGE,Instituição,População,Conta,Despesas Empenhadas,Despesas Liquidadas,Despesas Pagas,Inscrição de Restos a Pagar Não Processados,Inscrição de Restos a Pagar Processados,taxa_execucao_pct,empenhado_per_capita,pago_per_capita,saldo_nao_pago
0,2020,AC,1200401,Prefeitura Municipal de Rio Branco - AC,407319,01 - Legislativa,"29.014.059,54","28.264.216,80","28.264.216,80","749.842,74","0,00","97,42","71,23","69,39","749.842,74"
1,2020,AC,1200401,Prefeitura Municipal de Rio Branco - AC,407319,03 - Essencial à Justiça,"18.822.895,07","18.822.895,07","18.818.545,07","0,00","4.350,00","99,98","46,21","46,20","4.350,00"
2,2020,AC,1200401,Prefeitura Municipal de Rio Branco - AC,407319,04 - Administração,"82.258.800,45","81.364.049,25","81.360.049,25","894.751,20","4.000,00","98,91","201,95","199,75","898.751,20"
3,2020,AC,1200401,Prefeitura Municipal de Rio Branco - AC,407319,06 - Segurança Pública,"3.304.806,91","3.304.806,91","3.304.806,91","0,00","0,00","100,00","8,11","8,11","0,00"
4,2020,AC,1200401,Prefeitura Municipal de Rio Branco - AC,407319,08 - Assistência Social,"22.908.000,00","21.976.380,65","21.976.380,65","931.619,35","0,00","95,93","56,24","53,95","931.619,35"


## Ranking por função

Aqui a ideia é olhar para Saúde e Educação como exemplos centrais do desafio. Eu uso 2024 porque é o último ano completo da base e comparo as capitais pelo gasto empenhado, pago, taxa de execução e valores per capita.

In [6]:
def ranking_funcao(df, funcao, ano=2024, top_n=10):
    recorte = (
        df.loc[(df["Ano"] == ano) & (df["Conta"] == funcao)]
        .sort_values(by="Despesas Empenhadas", ascending=False)
        .loc[:, [
            "Instituição",
            "UF",
            "População",
            "Despesas Empenhadas",
            "Despesas Pagas",
            "taxa_execucao_pct",
            "empenhado_per_capita",
            "pago_per_capita",
        ]]
        .head(top_n)
    )
    return recorte.assign(Funcao=funcao)


ranking_saude = ranking_funcao(execucao_funcoes, "10 - Saúde")
ranking_educacao = ranking_funcao(execucao_funcoes, "12 - Educação")

pd.concat([ranking_saude, ranking_educacao], ignore_index=True)

Coluna,Instituição,UF,População,Despesas Empenhadas,Despesas Pagas,taxa_execucao_pct,empenhado_per_capita,pago_per_capita,Funcao
0,Prefeitura Municipal de São Paulo - SP,SP,12200180,"22.752.837.820,49","21.854.464.654,11","96,05","1.864,96","1.791,32",10 - Saúde
1,Prefeitura Municipal do Rio de Janeiro - RJ,RJ,6625849,"7.642.644.288,09","7.093.295.406,04","92,81","1.153,46","1.070,55",10 - Saúde
2,Prefeitura Municipal de Belo Horizonte - MG,MG,2392678,"5.993.532.028,34","5.391.129.586,40","89,95","2.504,95","2.253,18",10 - Saúde
3,Prefeitura Municipal de Fortaleza - CE,CE,2596157,"3.698.473.661,24","3.508.508.267,82","94,86","1.424,60","1.351,42",10 - Saúde
4,Prefeitura Municipal de Curitiba - PR,PR,1871789,"3.113.890.382,34","3.029.796.239,02","97,30","1.663,59","1.618,66",10 - Saúde
5,Prefeitura Municipal de Salvador - BA,BA,2610987,"2.885.482.525,11","2.861.717.678,29","99,18","1.105,13","1.096,03",10 - Saúde
6,Prefeitura Municipal de Porto Alegre - RS,RS,1404269,"2.799.019.820,54","2.443.202.864,91","87,29","1.993,22","1.739,84",10 - Saúde
7,Prefeitura Municipal de Goiânia - GO,GO,1414483,"2.157.573.979,74","2.068.845.703,41","95,89","1.525,34","1.462,62",10 - Saúde
8,Prefeitura Municipal de Campo Grande - MS,MS,942140,"1.984.757.017,22","1.750.535.980,59","88,20","2.106,65","1.858,04",10 - Saúde
9,Prefeitura Municipal de Recife - PE,PE,1494586,"1.937.953.959,80","1.910.772.845,55","98,60","1.296,65","1.278,46",10 - Saúde


## Maceió versus média das capitais

Além do ranking absoluto, eu comparo Maceió com a média das capitais em Saúde e Educação para ver se o comportamento local foge do padrão do conjunto.

In [7]:
funcoes_foco = ["10 - Saúde", "12 - Educação"]

maceio_funcoes = (
    execucao_funcoes.loc[
        execucao_funcoes["Instituição"].str.contains("Maceió", case=False, na=False)
        & execucao_funcoes["Conta"].isin(funcoes_foco)
    ]
    .groupby(["Ano", "Conta"], as_index=False)
    .agg(
        **{
            "Despesas Empenhadas": ("Despesas Empenhadas", "sum"),
            "Despesas Pagas": ("Despesas Pagas", "sum"),
            "taxa_execucao_pct": ("taxa_execucao_pct", "mean"),
            "empenhado_per_capita": ("empenhado_per_capita", "mean"),
            "pago_per_capita": ("pago_per_capita", "mean"),
        }
    )
)

media_capitais = (
    execucao_funcoes.loc[execucao_funcoes["Conta"].isin(funcoes_foco)]
    .groupby(["Ano", "Conta"], as_index=False)
    .agg(
        **{
            "media_despesas_empenhadas": ("Despesas Empenhadas", "mean"),
            "media_despesas_pagas": ("Despesas Pagas", "mean"),
            "media_taxa_execucao_pct": ("taxa_execucao_pct", "mean"),
            "media_empenhado_per_capita": ("empenhado_per_capita", "mean"),
            "media_pago_per_capita": ("pago_per_capita", "mean"),
        }
    )
)

comparativo_maceio = maceio_funcoes.merge(media_capitais, on=["Ano", "Conta"], how="left")
comparativo_maceio

,Ano,Conta,Despesas Empenhadas,Despesas Pagas,taxa_execucao_pct,empenhado_per_capita,pago_per_capita,media_despesas_empenhadas,media_despesas_pagas,media_taxa_execucao_pct,media_empenhado_per_capita,media_pago_per_capita
0,2020,10 - Saúde,"792.065.782,48","781.472.702,46","98,66","777,34","766,94","1.777.960.094,68","1.637.067.861,31","92,91","951,38","881,20"
1,2020,12 - Educação,"357.868.315,31","319.093.839,56","89,17","351,21","313,16","1.368.538.022,25","1.198.223.422,70","93,47","662,54","617,60"
2,2021,10 - Saúde,"774.134.174,08","755.985.079,24","97,66","754,99","737,29","1.928.151.359,78","1.784.537.408,93","92,71","1.013,51","939,08"
3,2021,12 - Educação,"464.266.926,96","384.922.142,66","82,91","452,78","375,40","1.656.466.803,39","1.353.859.693,42","85,52","774,61","659,03"
4,2022,10 - Saúde,"872.967.088,35","858.715.266,40","98,37","846,23","832,41","2.123.261.006,06","1.985.702.746,59","92,23","1.079,27","997,61"
5,2022,12 - Educação,"768.448.799,82","724.440.479,18","94,27","744,91","702,25","1.974.102.543,71","1.662.072.524,38","87,96","948,49","835,78"
6,2023,10 - Saúde,"1.302.781.725,57","1.274.498.167,54","97,83","1.262,88","1.235,46","2.407.263.745,86","2.252.488.382,45","93,30","1.247,08","1.158,78"
7,2023,12 - Educação,"648.604.839,79","612.559.556,10","94,44","628,74","593,80","2.092.079.592,17","1.898.898.490,54","91,32","1.050,91","957,60"
8,2024,10 - Saúde,"1.297.135.226,50","1.262.955.548,42","97,36","1.350,24","1.314,67","2.711.698.392,44","2.559.943.682,46","94,24","1.439,65","1.348,42"
9,2024,12 - Educação,"803.988.514,26","687.560.500,43","85,52","836,91","715,71","2.373.515.725,23","2.228.294.562,18","92,85","1.243,85","1.158,05"


## Subfunções mais relevantes

Para aprofundar a leitura, eu concentro o gasto nas subfunções mais importantes de Saúde e Educação, sempre usando apenas linhas analíticas para não duplicar totais. A lógica de `Tipo Conta` vem do ETL, não deste notebook.

In [8]:
execucao_subfuncoes = (
    base_subfuncoes.pivot_table(
        index=["Ano", "UF", "Cod.IBGE", "Instituição", "População", "Conta"],
        columns="Coluna",
        values="Valor",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for coluna in [
    "Despesas Empenhadas",
    "Despesas Liquidadas",
    "Despesas Pagas",
]:
    if coluna not in execucao_subfuncoes.columns:
        execucao_subfuncoes[coluna] = 0

execucao_subfuncoes["taxa_execucao_pct"] = (
    execucao_subfuncoes["Despesas Pagas"]
    .div(execucao_subfuncoes["Despesas Empenhadas"])
    .where(execucao_subfuncoes["Despesas Empenhadas"] > 0)
    * 100
)

sub_saude = (
    execucao_subfuncoes.loc[execucao_subfuncoes["Conta"].str.startswith("10.")]
    .groupby("Conta", as_index=False)
    .agg(
        **{
            "Despesas Empenhadas": ("Despesas Empenhadas", "sum"),
            "Despesas Pagas": ("Despesas Pagas", "sum"),
            "taxa_execucao_pct": ("taxa_execucao_pct", "mean"),
        }
    )
    .sort_values(by="Despesas Empenhadas", ascending=False)
    .head(10)
)

sub_educacao = (
    execucao_subfuncoes.loc[execucao_subfuncoes["Conta"].str.startswith("12.")]
    .groupby("Conta", as_index=False)
    .agg(
        **{
            "Despesas Empenhadas": ("Despesas Empenhadas", "sum"),
            "Despesas Pagas": ("Despesas Pagas", "sum"),
            "taxa_execucao_pct": ("taxa_execucao_pct", "mean"),
        }
    )
    .sort_values(by="Despesas Empenhadas", ascending=False)
    .head(10)
)

sub_saude.assign(Funcao="10 - Saúde").pipe(
    lambda primeira: pd.concat(
        [primeira, sub_educacao.assign(Funcao="12 - Educação")],
        ignore_index=True,
    )
)

,Conta,Despesas Empenhadas,Despesas Pagas,taxa_execucao_pct,Funcao
0,10.302 - Assistência Hospitalar e Ambulatorial,"145.860.093.390,27","133.803.195.406,28","90,49",10 - Saúde
1,10.301 - Atenção Básica,"80.827.158.805,39","77.294.980.825,03","93,43",10 - Saúde
2,10.122 - Administração Geral,"41.927.251.021,23","40.529.333.439,16","93,29",10 - Saúde
3,10.303 - Suporte Profilático e Terapêutico,"5.506.693.172,04","4.555.015.829,39","78,00",10 - Saúde
4,10.305 - Vigilância Epidemiológica,"4.463.636.225,07","4.202.545.013,15","92,95",10 - Saúde
5,10.304 - Vigilância Sanitária,"2.297.564.206,71","2.038.697.669,23","89,70",10 - Saúde
6,10.306 - Alimentação e Nutrição,"11.421.526,80","8.784.845,90","73,11",10 - Saúde
7,12.361 - Ensino Fundamental,"127.819.673.291,63","114.135.747.703,34","90,66",12 - Educação
8,12.365 - Educação Infantil,"85.254.506.542,64","76.167.901.793,28","89,95",12 - Educação
9,12.122 - Administração Geral,"8.725.372.228,03","7.971.734.202,81","90,83",12 - Educação


## Conclusão

A análise mostra que a comparação mais justa entre capitais passa por três filtros: usar apenas anos completos, separar função de subfunção e olhar o gasto por habitante, além do valor absoluto. Sem isso, capitais grandes dominam a leitura e as conclusões ficam distorcidas.

Em 2024, São Paulo, Rio de Janeiro e Belo Horizonte lideram os rankings absolutos em Saúde e Educação. Maceió aparece com comportamento diferente quando a métrica passa para execução financeira e valor per capita, o que reforça a importância de não depender só do total empenhado.